In [17]:
import matplotlib.pyplot as plt
import matplotlib.cm as mplcm
import matplotlib.colors as colors
import numpy as np
import napari
import tifffile
import pandas as pd
from napari.utils.colormaps import DirectLabelColormap
from skimage.transform import rescale
ALPHA = 0.4
TRANSPARENT = np.array([0.0, 0.0, 0.0, 0.0], dtype=float)

In [10]:
full_seg_mask = tifffile.imread(r"Z:\Bel\Jorge_SPACEFISH_Examples\Outputs\results\dev8_1_reg2\dev8_1_reg2-seg_mask.tif")
small_seg_mask = rescale(full_seg_mask, 0.25, order=0, anti_aliasing=False, preserve_range=True).astype(np.int64)
decoding_csv = pd.read_csv(r"Z:\Bel\Jorge_SPACEFISH_Examples\Outputs\results\dev8_1_reg2\dev8_1_reg2_spots_decoded.csv")

In [ ]:
unique_genes = decoding_csv.gene.unique().tolist()
filtered_genes = list(dict.fromkeys("FP" if "FP" in gene else gene for gene in unique_genes))
filtered_genes.sort()

['ANGPT2', 'APLN', 'CDH5', 'COL1A1', 'COL1A2', 'DLL4', 'FLT1', 'FP', 'ICAM1', 'ITGA6', 'JAG1', 'KDR', 'MMP1', 'PDGFB', 'PDGFRB', 'PECAM1', 'SEMA3F', 'VEGFA', 'VWF']


In [25]:
cm_genes = plt.get_cmap("gist_rainbow")
gene_norm = colors.Normalize(vmin=0, vmax=max(len(filtered_genes) - 1, 1))
gene_smap = mplcm.ScalarMappable(norm=gene_norm, cmap=cm_genes)
gene_colours = {g: gene_smap.to_rgba(i) for i, g in enumerate(sorted(filtered_genes))}
gene_to_nuclei = (decoding_csv.groupby("gene")["nucleus"].apply(set).to_dict())


In [27]:
viewer = napari.Viewer()

mask_int = small_seg_mask.astype(np.int64)
all_labels = np.unique(mask_int)
all_labels = all_labels[all_labels != 0]

for gene in filtered_genes:
    r, g, b, _ = gene_colours[gene]
    gene_rgba = np.array([r, g, b, ALPHA], dtype=float)
    nuclei = gene_to_nuclei.get(gene, set())

    # Map every label to either this gene's colour (if it expresses it) or transparent.
    color_dict = {None: TRANSPARENT, 0: TRANSPARENT}
    for lab in all_labels:
        color_dict[int(lab)] = gene_rgba if int(lab) in nuclei else TRANSPARENT

    viewer.add_labels(
        mask_int,
        name=f"gene:{gene}",
        colormap=DirectLabelColormap(color_dict=color_dict),
        blending="translucent",
    )
